# Adaptive Grover search in native FlagQuantum

Search two Boolean SAT instances and a graph partition problem with native Python `qfunc` programs. Each attempt synthesizes a checked static repetition count, executes a quantum state and draws 1000 observations. The classical loop inspects observed assignments and increases the Grover power only when no marked **individual row** strictly exceeds the threshold.

The quantum kernels use uniform preparation, coherent predicate phases and phase-exact diffusion. The existing four- and five-qubit native examples are reused. No source Qmod or remote service is loaded.


In [ ]:
from matplotlib import pyplot as plt

import flagquantum as fq
from examples.synthesis.algorithms.search_and_optimization.grover.grover.overview.workflow import (
    build_program,
    plot_graph,
    run_search,
    truth_table,
)


## Three-variable SAT

The first variable is logical `q[0]`, or packed integer bit zero. The truth table lists logical variables in source order. This six-clause predicate has two solutions; the later three-row display limit is not a solution-count assertion.


In [ ]:
for bits, marked in truth_table("small_3sat"):
    print(bits, "marked:", marked)
small_program = build_program("small_3sat", 1)
print(small_program.report["resources"])
print(fq.Circuit.from_ir(small_program.circuit).draw())
small = run_search("small_3sat", seed=1100)
for attempt in small.attempts:
    print(
        "Loop / repetitions / seed:",
        attempt.loop_index,
        attempt.repetitions,
        attempt.seed,
    )
print("Stopped with threshold exceeded:", small.succeeded)
for row in small.attempts[-1].outcomes[:3]:
    print(row)


## Four-variable SAT

Preserve all thirteen original clauses, the same 0.45 strict per-row threshold and the five-row result display. Both SAT instances have two satisfying assignments. Uniform-subspace mass does not itself satisfy an individual-row stopping condition.


In [ ]:
for bits, marked in truth_table("large_3sat"):
    print(bits, "marked:", marked)
large_program = build_program("large_3sat", 1)
print(large_program.report["resources"])
print(fq.Circuit.from_ir(large_program.circuit).draw())
large = run_search("large_3sat", seed=1200)
for attempt in large.attempts:
    print(
        "Loop / repetitions / seed:",
        attempt.loop_index,
        attempt.repetitions,
        attempt.seed,
    )
print("Stopped with threshold exceeded:", large.succeeded)
for row in large.attempts[-1].outcomes[:5]:
    print(row)


## Five-vertex MaxCut

Use the original six-edge planar graph. Mark assignments with cut size **at least four**. Use the source threshold 0.1 and display up to all 32 observed words, retaining count, probability and classical predicate. The most observed word controls the graph colors and crossing-edge widths, even if a different word triggered the stop.


In [ ]:
graph_program = build_program("max_cut", 1)
print(graph_program.report["resources"])
print(fq.Circuit.from_ir(graph_program.circuit).draw())
graph = run_search("max_cut", seed=1300)
for attempt in graph.attempts:
    print(
        "Loop / repetitions / seed:",
        attempt.loop_index,
        attempt.repetitions,
        attempt.seed,
    )
print("Stopped with threshold exceeded:", graph.succeeded)
for row in graph.attempts[-1].outcomes[:32]:
    print(row)


In [ ]:
figure = plot_graph(graph)
plt.show()
figure


## Bounded stopping and reproducibility

The source loop uses `ceil(pi/4 * sqrt(2**i))` for `i=0..3`, giving `1,2,2,3`. The duplicate consumes a loop iteration without another execution. If no marked row strictly exceeds its threshold, return the final actual attempt with `succeeded=False`. Setting the threshold to one demonstrates this failure path; equality never counts as success.

Native seeds 1100/1200/1300 plus the loop index are explicit reproducibility adaptations: the source remote sampler had no fixed shot seed. Named probability extraction removes manual wire decoding. The planar visualization, truth tables and display limits preserve the original workflow; local text circuit drawings and resource records replace the hosted viewer.


In [ ]:
not_found = run_search("small_3sat", threshold=1.0, seed=1100)
print("Succeeded:", not_found.succeeded)
print("Executed loop indices:", [attempt.loop_index for attempt in not_found.attempts])
print("Executed repetitions:", [attempt.repetitions for attempt in not_found.attempts])
assert not not_found.succeeded
assert [attempt.loop_index for attempt in not_found.attempts] == [0, 1, 3]
